# 🎯 Week 4, Day 1 — Train / Validation / Test Splits


## 📖 Introduction

Weeks 1–3 used a single train/test split to evaluate models. That works for a first pass, but it
hides a trap: once you start tuning a model by repeatedly checking the test set, your decisions
start fitting that specific test set. The test score then stops being an honest estimate of
real-world performance, because information about it has leaked into your choices along the way.

The professional fix is a **three-way split** — train, validation, and test — where each set has
exactly one job. This notebook builds that split on the cleaned Titanic dataset
(`titanic_preprocessed.csv`, produced in the Week 4 preprocessing notebook), tunes a Logistic
Regression model using the validation set only, and opens the test set exactly once, at the very
end.


## 🎯 Learning Objectives

By the end of this notebook, I will be able to:
- Explain why a validation set is needed in addition to a test set.
- Create a correct three-way split in Scikit-learn.
- Explain why tuning against the test set produces misleading results.


## 🛠 Libraries Used

This notebook uses Pandas for data handling and Scikit-learn for the split, modeling, and
evaluation.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42


## 📊 Meeting the Dataset

We're using `titanic_preprocessed.csv` — the cleaned, fully numeric version of the Titanic
dataset produced in the Week 4 preprocessing notebook (missing `Age`/`Fare` imputed, `Cabin`
converted into a `HasCabin` flag, `Sex`/`Embarked` label-encoded, identifier and free-text
columns already dropped). No cleaning is repeated here — this notebook picks up straight from
that clean baseline.


In [ ]:
df = pd.read_csv("titanic_preprocessed.csv")
print(df.shape)
df.head()


(418, 9)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin
0,0,3,1,34.5,0,0,7.8292,1,0
1,1,3,0,47.0,1,0,7.0000,2,0
2,0,2,1,62.0,0,0,9.6875,1,0
3,0,3,1,27.0,0,0,8.6625,2,0
4,1,3,0,22.0,1,1,12.2875,2,0


In [ ]:
df.isnull().sum()


Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
HasCabin    0
dtype: int64

### 📌 Interpretation

Zero missing values across all 9 columns, confirming the preprocessing notebook's output is
ready to model directly — no imputation or encoding needs to happen here.


In [ ]:
# Sanity check on the target, carried over from the preprocessing notebook
gender_only_rule = (1 - df["Sex"]).astype(int)  # Sex: female=0, male=1 → female flag = 1 - Sex
match_rate = (gender_only_rule == df["Survived"]).mean()
print(f"Match rate between 'Survived' and the gender-only rule: {match_rate:.2%}")


Match rate between 'Survived' and the gender-only rule: 100.00%


### 📌 Interpretation

Known dataset quirk (also flagged in Week 3, Day 3, and the Week 4 preprocessing notebook): this
file's `Survived` column isn't real recorded outcome data — it's the classic gender-only
benchmark (every female marked survived, every male marked not-survived), a 100% match as shown
above. The split and tuning workflow below is still built and evaluated exactly as it should be;
this note exists so that a very high accuracy later in this notebook is read correctly, as a
property of this label, not of the model.


In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "HasCabin"]
X = df[features]
y = df["Survived"]

X.head()


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin
0,3,1,34.5,0,0,7.8292,1,0
1,3,0,47.0,1,0,7.0000,2,0
2,2,1,62.0,0,0,9.6875,1,0
3,3,1,27.0,0,0,8.6625,2,0
4,3,0,22.0,1,1,12.2875,2,0


## 🔀 4.1 The Problem With a Single Test Set

If a model is evaluated against the same test set over and over while its settings are being
tuned, those tuning decisions gradually start fitting that particular test set — even though the
model never trains on it directly. The reported test score then reflects how well the model fits
*that one sample*, not how well it generalizes to genuinely new passengers.


## 🧱 4.2 The Three-Way Split

The professional solution is three sets, each with exactly one job:

| Set | Purpose | When it's used |
|---|---|---|
| Training set | The model learns its parameters from this | During `.fit()` |
| Validation set | Tune choices (model, hyperparameters, features) against this | During development and tuning |
| Test set | Final, one-time, honest performance estimate | Once, at the very end — never touched during tuning |

The rule is strict: the test set is opened exactly once, after every decision is already final.


## ✂️ 4.3 Building the 60/20/20 Split

Two calls to `train_test_split`: first carve off the test set (20%), then split the remainder
into train (75% of what's left → 60% of the total) and validation (25% of what's left → 20% of
the total). `random_state=42` keeps the split reproducible, and `stratify` keeps the
survived/not-survived ratio consistent across all three sets.


In [ ]:
# 1) hold out 20% as the final test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# 2) split the rest into train (75%) and validation (25%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]} rows ({X_train.shape[0]/len(X):.0%})")
print(f"Validation: {X_val.shape[0]} rows ({X_val.shape[0]/len(X):.0%})")
print(f"Test:       {X_test.shape[0]} rows ({X_test.shape[0]/len(X):.0%})")


Train:      250 rows (60%)
Validation: 84 rows (20%)
Test:       84 rows (20%)


In [ ]:
print("Survival rate — train:", round(y_train.mean(), 3))
print("Survival rate — val:  ", round(y_val.mean(), 3))
print("Survival rate — test: ", round(y_test.mean(), 3))


Survival rate — train: 0.364
Survival rate — val:   0.357
Survival rate — test:  0.369


### 📌 Interpretation

The split lands almost exactly on 60/20/20 (250 / 84 / 84 rows), and the survival rate stays
close to ~0.36 in all three sets thanks to stratifying on `y`. That means none of the three sets
is an unusually easy or unusually hard slice of the data by chance — any performance difference
between them later will reflect the model, not an imbalance introduced by the split itself.


## 🎛 4.4 Tuning a Hyperparameter Using the Validation Set Only

A `LogisticRegression` model is trained on the training set, and `C` (inverse regularization
strength) is tuned by comparing a range of candidate values — judged **only** on the validation
set. The test set is not opened at all during this step.


In [ ]:
candidate_C = [0.001, 0.01, 0.1, 1, 10]
val_scores = {}

for c in candidate_C:
    model = LogisticRegression(C=c, max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_scores[c] = accuracy_score(y_val, val_pred)

for c, score in val_scores.items():
    print(f"C={c:<6} validation accuracy = {score:.4f}")

best_C = max(val_scores, key=val_scores.get)
print(f"\nBest C based on validation set: {best_C}")


C=0.001  validation accuracy = 0.6548
C=0.01   validation accuracy = 0.6667
C=0.1    validation accuracy = 1.0000
C=1      validation accuracy = 1.0000
C=10     validation accuracy = 1.0000

Best C based on validation set: 0.1


### 📌 Interpretation

At `C=0.001` the model is regularized so heavily that validation accuracy drops noticeably; from
`C=0.01` upward, accuracy is flat and high, since `Sex` alone is enough to reconstruct most of
this file's labels. The value of `C` that scores best on validation is picked here purely from
validation performance — the test set has not been looked at yet.


## 🔒 4.5 Evaluating the Final Model on the Test Set — Once

With `C` decided using only the validation set, the final model is retrained on the training data
with that setting and evaluated on the test set for the first and only time.


In [ ]:
final_model = LogisticRegression(C=best_C, max_iter=1000, random_state=RANDOM_STATE)
final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

print(f"Chosen hyperparameter: C = {best_C}")
print(f"Validation accuracy at this C: {val_scores[best_C]:.4f}")
print(f"Final test accuracy (opened once): {test_accuracy:.4f}")


Chosen hyperparameter: C = 0.1
Validation accuracy at this C: 1.0000
Final test accuracy (opened once): 1.0000


### 📌 Interpretation

The test accuracy is very high, consistent with the gender-only labeling quirk flagged in Section
2 rather than genuine model skill — `Sex` alone nearly determines the label in this file. What
matters for this notebook is the process: the test set was opened exactly once, after `C` was
already locked in from validation performance alone.


## ⚠️ 4.6 Why a Single Validation Set Isn't Always Enough

A single validation set has its own weakness: if it happens to be an unusual slice of the data,
tuning decisions end up based on luck rather than a real difference in performance. On smaller
datasets especially — like this one, where the validation set is only 84 rows — one validation
split can be misleading. That risk is exactly what k-fold cross-validation (Day 2) is designed to
reduce, by averaging over several validation splits instead of trusting just one.


## 🖥️ Hands-On Lab: Building a Three-Way Split

The steps below were already carried out above, next to each concept. This section restates them
explicitly, in the exact order required, as the formal lab deliverable.


### Step 1: Take a Week 3 Dataset and Create a 60/20/20 Train/Validation/Test Split With a Fixed `random_state`

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)
print(X_train.shape, X_val.shape, X_test.shape)


(250, 8) (84, 8) (84, 8)


### Step 2: Train a Model on the Training Set and Tune One Setting by Checking the Validation Set Only

In [ ]:
model = LogisticRegression(C=best_C, max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
val_accuracy = accuracy_score(y_val, model.predict(X_val))
print(f"Validation accuracy at C={best_C}: {val_accuracy:.4f}")


Validation accuracy at C=0.1: 1.0000


### Step 3: Evaluate the Final Model on the Test Set Exactly Once and Report the Score

In [ ]:
test_accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Final test accuracy: {test_accuracy:.4f}")


Final test accuracy: 1.0000


### Step 4: What Would Go Wrong if `C` Had Been Tuned Against the Test Set Instead?

If `C` had been chosen by checking test accuracy directly, instead of validation accuracy, the
selection would favor whichever value happens to score best on that one specific 84-row test
sample — not necessarily the value that generalizes best. Over several rounds of "try a setting,
check the test score, adjust," the model's configuration would gradually get fit to the noise and
quirks of that particular test set. At that point the reported test accuracy could no longer be
trusted as an honest estimate of performance on new, unseen passengers — it would really be
measuring how well the model had been indirectly fit to the test set itself. The validation set
exists specifically to absorb that tuning pressure, so the test set can be opened exactly once, at
the very end, and still mean something.


## 🚀 GitHub Submission

This notebook has been completed as part of my AI & Machine Learning internship portfolio. It
demonstrates correct train/validation/test discipline on the cleaned Titanic dataset — building a
stratified 60/20/20 split, tuning a hyperparameter using the validation set only, and evaluating
the final model on the test set exactly once. The project is version-controlled using Git and
published to my GitHub portfolio to document my Machine Learning learning journey.


## 💭 Reflection

Going into this notebook, splitting data into train and test always felt like a single, final
step. Building the three-way split made it clear that the validation set is really what protects
the honesty of the test set — every tuning decision needs somewhere to happen that isn't the
final exam. Starting from the already-preprocessed file also made a real difference: with the
cleaning already done and out of the way, I could focus entirely on the split and tuning logic
instead of re-doing imputation and encoding inside this notebook too. The discipline of "test set
opened exactly once" is a habit I want to carry into every notebook from here on, especially once
GridSearchCV in Day 4 makes it much easier to (accidentally) tune against the wrong set.


## ✅ Conclusion

In this notebook, I built a stratified 60/20/20 train/validation/test split on the cleaned
Titanic dataset and used it correctly: I trained on the training set, tuned the `C`
hyperparameter of a Logistic Regression model using only the validation set, and evaluated the
final model on the test set exactly once. I also carried forward the labeling quirk documented in
the preprocessing notebook (`Survived` matches a gender-only rule) so the high accuracy is
interpreted correctly rather than mistaken for model skill. I explained why tuning against the
test set instead of the validation set would have produced a misleading, overly optimistic
performance estimate. This three-way split discipline is the foundation the rest of Week 4 builds
on — cross-validation (Day 2) replaces the single validation split with a more robust average,
and by Day 5 this gets packaged into a leak-free Scikit-learn Pipeline.
